In [3]:
import sys, os
from pathlib import Path

# Resolve project root regardless of where Jupyter was launched from
_root = Path.cwd()
if not (_root / 'src').exists():
    _root = _root.parent
sys.path.insert(0, str(_root / 'src'))

from ingestor import load_locomo
from memory_writer import MemoryWriter
from memory_store import MemoryStore
from context_builder import ContextBuilder, RetrievalConfig
from evaluator import answer_question

from openai import OpenAI
from sentence_transformers import SentenceTransformer

# Initialize
client = OpenAI()  # uses OPENAI_API_KEY env var
embedder = SentenceTransformer('all-MiniLM-L6-v2')
print('Initialized.')

c:\Users\evag3\Desktop\code\semantic-caching\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4573.80it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Initialized.


In [4]:
# Load dataset
conversations = load_locomo(_root / 'data' / 'locomo10.json')
conv = conversations[0]
print(f'Working with session: {conv.session_id}')
print(f'Turns: {len(conv.turns)}, QA pairs: {len(conv.qa_pairs)}')

Loaded 10 conversations from c:\Users\evag3\Desktop\code\semantic-caching\data\locomo10.json
Working with session: conv-26
Turns: 419, QA pairs: 199


In [5]:
# Load dataset
conversations = load_locomo('../data/locomo10.json')
conv = conversations[0]
print(f'Working with session: {conv.session_id}')
print(f'Turns: {len(conv.turns)}, QA pairs: {len(conv.qa_pairs)}')

Loaded 10 conversations from ..\data\locomo10.json
Working with session: conv-26
Turns: 419, QA pairs: 199


In [6]:
# ── Step 1: Extract memories ──────────────────
writer = MemoryWriter(
    client=client,
    embedder=embedder,
    window_size=4,
    stride=2,
    importance_threshold=0.3,
    dedup_threshold=0.92,
)

print('Extracting memories (this calls OpenAI API)...')
memories = writer.process_conversation(conv, verbose=True)
print(f'\nExtracted {len(memories)} memories')

Extracting memories (this calls OpenAI API)...
[MemoryWriter] session=conv-26: 419 turns → 687 raw → 516 after dedup

Extracted 516 memories


In [7]:
# Inspect extracted memories
print('Top 10 memories by importance:\n')
for m in sorted(memories, key=lambda x: x.importance, reverse=True)[:10]:
    print(f'  [{m.type:12s}] importance={m.importance:.2f}  persistence={m.persistence}')
    print(f'  "{m.fact}"')
    print(f'  source_turns={m.source_turns}')
    print()

Top 10 memories by importance:

  [constraint  ] importance=1.00  persistence=long_term
  "Caroline is up for the challenge of being a single parent."
  source_turns=[30, 31, 32, 33]

  [goal        ] importance=0.90  persistence=long_term
  "Caroline's goal is to make sure the kids have a safe and loving home."
  source_turns=[30, 31, 32, 33]

  [decision    ] importance=0.90  persistence=medium
  "Melanie agrees to plan something special."
  source_turns=[246, 247, 248, 249]

  [goal        ] importance=0.90  persistence=long_term
  "The LGBTQ art show aims to spread understanding and acceptance."
  source_turns=[302, 303, 304, 305]

  [goal        ] importance=0.90  persistence=long_term
  "The goal of Caroline's show is to spread understanding and acceptance."
  source_turns=[304, 305, 306, 307]

  [goal        ] importance=0.90  persistence=long_term
  "Caroline's dream is to adopt and provide a safe, loving home for kids who need it."
  source_turns=[354, 355, 356, 357]

  [goal 

In [8]:
# ── Step 2: Store in Chroma ───────────────────
store = MemoryStore(collection_name='demo', embedder=embedder)
store.add_batch(memories)
print(f'Stored {store.count()} memories in Chroma')

Stored 516 memories in Chroma


In [9]:
# ── Step 3: Retrieve for a query ─────────────
config = RetrievalConfig(
    alpha=0.5, beta=0.3, gamma=0.1, lam=0.1,
    token_budget=500,
)
builder = ContextBuilder(store=store, config=config)

# Use first QA pair as example
qa = conv.qa_pairs[0]
print(f'Question: {qa.question}')
print(f'Gold answer: {qa.answer}')
print(f'Evidence turns: {qa.evidence_turn_ids}')
print()

context, debug = builder.build_context(
    query=qa.question,
    session_id=conv.session_id,
    verbose=True,
)
print()
print('=== Retrieved Context ===')
print(context)

Question: When did Caroline go to the LGBTQ support group?
Gold answer: 7 May 2023
Evidence turns: [2]

[ContextBuilder] 50 candidates → 31 selected (753 tokens)

=== Retrieved Context ===
[MEMORY CONTEXT]
1. [FACT] Caroline is putting together an LGBTQ art show next month.  (importance=0.80)
2. [GOAL] Caroline uses her art to speak up for the LGBTQ+ community and push for acceptance.  (importance=0.80)
3. [GOAL] The pride parade inspired Caroline to keep fighting for LGBTQ rights.  (importance=0.80)
4. [GOAL] Caroline wants to help make a difference in LGBTQ rights.  (importance=0.80)
5. [FACT] The group Caroline joined is called 'Connected LGBTQ Activists'.  (importance=0.60)
6. [FACT] Caroline finds fulfillment in her involvement with the activist group.  (importance=0.70)
7. [FACT] Caroline's show will feature LGBTQ artists and their talents.  (importance=0.60)
8. [FACT] Hearing others share and celebrate their identities was empowering for Caroline.  (importance=0.60)
9. [FACT] Ca

In [10]:
# Scoring breakdown
print('Scoring breakdown for retrieved memories:\n')
for d in debug:
    print(f'  score={d["score"]:.3f}  rel={d["relevance"]:.3f}  '
          f'imp={d["importance"]:.2f}  rec={d["recency"]:.2f}')
    print(f'  "{d["fact"][:80]}"')
    print()

Scoring breakdown for retrieved memories:

  score=0.748  rel=0.867  imp=0.80  rec=0.74
  "Caroline is putting together an LGBTQ art show next month."

  score=0.728  rel=0.866  imp=0.80  rec=0.55
  "Caroline uses her art to speak up for the LGBTQ+ community and push for acceptan"

  score=0.722  rel=0.853  imp=0.80  rec=0.55
  "The pride parade inspired Caroline to keep fighting for LGBTQ rights."

  score=0.720  rel=0.902  imp=0.80  rec=0.29
  "Caroline wants to help make a difference in LGBTQ rights."

  score=0.693  rel=0.928  imp=0.60  rec=0.49
  "The group Caroline joined is called 'Connected LGBTQ Activists'."

  score=0.686  rel=0.854  imp=0.70  rec=0.49
  "Caroline finds fulfillment in her involvement with the activist group."

  score=0.684  rel=0.859  imp=0.60  rec=0.74
  "Caroline's show will feature LGBTQ artists and their talents."

  score=0.681  rel=0.819  imp=0.60  rec=0.91
  "Hearing others share and celebrate their identities was empowering for Caroline."

  score=0.

In [11]:
# ── Step 4: Answer the question ───────────────
predicted = answer_question(client, qa.question, context)
print(f'Question:  {qa.question}')
print(f'Gold:      {qa.answer}')
print(f'Predicted: {predicted}')

Question:  When did Caroline go to the LGBTQ support group?
Gold:      7 May 2023
Predicted: I don't have enough information to answer that.


In [12]:
# ── Step 5: Score it ─────────────────────────
from evaluator import exact_match, token_f1

em = exact_match(predicted, qa.answer)
f1 = token_f1(predicted, qa.answer)
print(f'Exact match: {em}')
print(f'Token F1:    {f1:.3f}')

Exact match: False
Token F1:    0.000


In [13]:
# ── Interactive: try your own query ──────────
my_query = 'What decisions were made about the project?'  # change this

context, _ = builder.build_context(query=my_query, session_id=conv.session_id)
answer = answer_question(client, my_query, context)
print(f'Q: {my_query}')
print(f'A: {answer}')
print()
print('Context used:')
print(context)

Q: What decisions were made about the project?
A: Melanie agreed to plan something special with Caroline for their project (Memories 4, 8). Additionally, Melanie will start thinking about what they can do (Memory 22).

Context used:
[MEMORY CONTEXT]
1. [GOAL] Caroline's goal is to have a family.  (importance=0.90)
2. [GOAL] The goal of Caroline's show is to spread understanding and acceptance.  (importance=0.90)
3. [GOAL] Caroline hopes to build her own family and put a roof over kids who haven't had that before.  (importance=0.80)
4. [DECISION] Melanie agrees to plan something special.  (importance=0.90)
5. [GOAL] Caroline was inspired to make some art after the event.  (importance=0.70)
6. [GOAL] Caroline hoped to make a difference by sharing her story and offering support.  (importance=0.80)
7. [GOAL] Caroline wants to make something similar to the rainbow sidewalk.  (importance=0.80)
8. [DECISION] Melanie agreed to plan something special with Caroline.  (importance=0.80)
9. [FACT] 